In [ ]:
!pip install chembl_webresource_client rdkit pandas torch scikit-learn tqdm catboost --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from chembl_webresource_client.new_client import new_client
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
from tqdm import tqdm
import warnings
import copy

warnings.filterwarnings('ignore')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Hardware Accelerator: {device}")


In [ ]:
def fetch_data(target_id="CHEMBL203", limit=10000):
    print(f"\n1️Fetching High-Quality Data for {target_id} (EGFR)...")
    prot_seq = "MRPSGTAGAALLALLAALCPASRALEEKKVCQGTSNKLTQLGTFEDHFLSLQRMFNNCEVVLGNLEITYVQRNYDLSFLKTIQEVAGYVLIALNTVERIPLENLQIIRGNMYYENSYALAVLSNYDANKTGLKELPMRNLQEILHGAVRFSNNPALCNVESIQWRDIVSSDFLSNMSMDFQNHLGSCQKCDPSCPNGSCWGAGEENCQKLTKIICAQQCSGRCRGKSPSDCCHNQCAAGCTGPRESDCLVCRKFRDEATCKDTCPPLMLYNPTTYQMDVNPEGKYSFGATCVKKCPRNYVVTDHGSCVRACGADSYEMEEDGVRKCKKCEGPCRKVCNGIGIGEFKDSLSINATNIKHFKNCTSISGDLHILPVAFRGDSFTHTPPLDPQELDILKTVKEITGFLLIQAWPENRTDLHAFENLEIIRGRTKQHGQFSLAVVSLNITSLGLRSLKEISDGDVIISGNKNLCYANTINWKKLFGTSGQKTKIISNRGENSCKATGQVCHALCSPEGCWGPEPRDCVSCRNVSRGRECVDKCNLLEGEPREFVENSECIQCHPECLPQAMNITCTGRGPDNCIQCAHYIDGPHCVKTCPAGVMGENNTLVWKYADAGHVCHLCHPNCTYGCTGPGLEGCPTNGPKIPSIATGMVGALLLLLVVALGIGLFMRRRHIVRKRTLRRLLQERELVEPLTPSGEAPNQALLRILKETEFKKIKVLGSGAFGTVYKGLWIPEGEKVKIPVAIKELREATSPKANKEILDEAYVMASVDNPHVCRLLGICLTSTVQLITQLMPFGCLLDYVREHKDNIGSQYLLNWCVQIAKGMNYLEDRRLVHRDLAARNVLVKTPQHVKITDFGLAKLLGAEEKEYHAEGGKVPIKWMALESILHRIYTHQSDVWSYGVTVWELMTFGSKPYDGIPASEISSILEKGERLPQPPICTIDVYMIMVKCWMIDADSRPKFRELIIEFSKMARDPQRYLVIQGDERMHLPSPTDSNFYRALMDEEDMDDVVDADEYLIPQQGFFSSPSTSRTPLLSSLSATSNNSTVACIDRNGLQSCPIKEDSFLQRYSSDPTGALTEDSIDDTFLPVPEYINQSVPKRPAGSVQNPVYHNQPLNPAPSRDPHYQDPHSTAVGNPEYLNTVQPTCVNSTFDSPAHWAQKGSHQISLDNPDYQQDFFPKEAKPNGIFKGSTAENAEYLRVAPQSSEFIGA"

    activity = new_client.activity
    res = activity.filter(target_chembl_id=target_id, type="IC50", relation="=").only(['canonical_smiles', 'standard_value'])

    data = []
    print("⏳ Filtering Data (Active < 100nM | Inactive > 10,000nM)...")

    count = 0
    for item in tqdm(res):
        if count >= limit: break
        try:
            smiles = item['canonical_smiles']
            val = float(item['standard_value'])
            if val <= 100: label = 1.0
            elif val >= 10000: label = 0.0
            else: continue

            mol = Chem.MolFromSmiles(smiles)
            if mol and len(smiles) < 150:
                data.append({'smiles': smiles, 'protein': prot_seq, 'label': label, 'mol': mol})
                count += 1
        except: continue

    df = pd.DataFrame(data)
    pos = df[df['label'] == 1.0]
    neg = df[df['label'] == 0.0]
    min_len = min(len(pos), len(neg))
    df_bal = pd.concat([pos.sample(min_len, random_state=42), neg.sample(min_len, random_state=42)])
    df_bal = df_bal.sample(frac=1, random_state=42).reset_index(drop=True)
    print(f"Final Dataset: {len(df_bal)} balanced samples.")
    return df_bal

df = fetch_data(limit=10000)

In [ ]:
print("\n2️ Generating Features...")

class CharTokenizer:
    def __init__(self, seqs, max_len):
        chars = list(set("".join(seqs)))
        self.char_to_int = {c: i+1 for i, c in enumerate(chars)}
        self.vocab_size = len(self.char_to_int) + 1
        self.max_len = max_len

    def encode(self, seq):
        seq = seq[:self.max_len]
        enc = [self.char_to_int.get(c, 0) for c in seq]
        return enc + [0]*(self.max_len - len(enc))

d_tok = CharTokenizer(df['smiles'].values, max_len=100)
p_tok = CharTokenizer(df['protein'].values, max_len=800)
print("   Generating High-Res Fingerprints (2048 bits)...")
mfgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fps = []
for mol in df['mol']:
    fp = mfgen.GetFingerprintAsNumPy(mol)
    fps.append(fp)
X_fps = np.array(fps)

In [ ]:
class DeepDTI(nn.Module):
    def __init__(self, d_vocab, p_vocab, embed_dim=128):
        super().__init__()

        self.d_emb = nn.Embedding(d_vocab, embed_dim)
        self.d_trans = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=embed_dim, nhead=4, batch_first=True, dropout=0.1),
            num_layers=2
        )

        self.p_emb = nn.Embedding(p_vocab, embed_dim)
        self.p_cnn = nn.Sequential(
            nn.Conv1d(embed_dim, embed_dim, 7, padding=3), nn.ReLU(),
            nn.MaxPool1d(2)
        )
        self.p_trans = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=embed_dim, nhead=4, batch_first=True, dropout=0.1),
            num_layers=2
        )

        self.cross_attn = nn.MultiheadAttention(embed_dim, num_heads=4, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(embed_dim, 1), nn.Sigmoid())

    def forward(self, d, p):
        d_x = self.d_trans(self.d_emb(d))

        p_x = self.p_emb(p).permute(0, 2, 1)
        p_cnn = self.p_cnn(p_x).permute(0, 2, 1)
        p_x = self.p_trans(p_cnn)

        attn, _ = self.cross_attn(d_x, p_x, p_x)
        return self.fc(attn.mean(dim=1)).squeeze()

    def get_embedding(self, d, p):
        with torch.no_grad():
            d_x = self.d_trans(self.d_emb(d))
            p_x = self.p_emb(p).permute(0, 2, 1)
            p_cnn = self.p_cnn(p_x).permute(0, 2, 1)
            p_x = self.p_trans(p_cnn)
            attn, _ = self.cross_attn(d_x, p_x, p_x)
            return attn.mean(dim=1).cpu().numpy()

In [ ]:
class HybridDataset(Dataset):
    def __init__(self, d, p, y, dt, pt):
        self.d, self.p, self.y = d, p, y
        self.dt, self.pt = dt, pt
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return (torch.tensor(self.dt.encode(self.d[i]), dtype=torch.long),
                torch.tensor(self.pt.encode(self.p[i]), dtype=torch.long),
                torch.tensor(self.y[i], dtype=torch.float32))
indices = np.arange(len(df))
train_idx, test_idx = train_test_split(indices, test_size=0.15, random_state=42)

train_ds = HybridDataset(df['smiles'].values[train_idx], df['protein'].values[train_idx], df['label'].values[train_idx], d_tok, p_tok)
test_ds = HybridDataset(df['smiles'].values[test_idx], df['protein'].values[test_idx], df['label'].values[test_idx], d_tok, p_tok)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

model = DeepDTI(d_tok.vocab_size, p_tok.vocab_size).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.0005)
criterion = nn.BCELoss()

def calc_acc(loader):
    model.eval()
    correct = 0; total = 0
    with torch.no_grad():
        for d, p, y in loader:
            d, p, y = d.to(device), p.to(device), y.to(device)
            out = model(d, p)
            preds = (out > 0.5).float()
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total

In [ ]:
print("-" * 65)
print(f"{'Epoch':<10} | {'Train Loss':<12} | {'Train Acc':<12} | {'Val Acc':<12}")
print("-" * 65)
best_val_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())

for epoch in range(30):
    model.train()
    running_loss = 0.0
    for d, p, y in train_loader:
        d, p, y = d.to(device), p.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(d, p)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    train_loss = running_loss / len(train_loader)
    train_acc = calc_acc(train_loader)
    val_acc = calc_acc(test_loader)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())

    print(f"{epoch+1:<10} | {train_loss:<12.4f} | {train_acc*100:<11.2f}% | {val_acc*100:<11.2f}%")

print("-" * 65)
print(f"Best DL Validation Accuracy: {best_val_acc*100:.2f}%")

model.load_state_dict(best_model_wts)

In [ ]:
def get_hybrid_features(idx_list):
    dl_feats = []
    temp_loader = DataLoader(
        HybridDataset(df['smiles'].values[idx_list], df['protein'].values[idx_list], df['label'].values[idx_list], d_tok, p_tok),
        batch_size=64, shuffle=False
    )
    for d, p, y in temp_loader:
        d, p = d.to(device), p.to(device)
        dl_feats.extend(model.get_embedding(d, p))
    dl_feats = np.array(dl_feats)
    fp_feats = X_fps[idx_list]

    return np.hstack([dl_feats, fp_feats]), df['label'].values[idx_list]

X_train, y_train = get_hybrid_features(train_idx)
X_test, y_test = get_hybrid_features(test_idx)
print("\nTraining CatBoost")
cb = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.03,
    depth=6,
    loss_function='Logloss',
    eval_metric='Accuracy',
    verbose=0,
    task_type="GPU" if torch.cuda.is_available() else "CPU"
)
cb.fit(X_train, y_train)
y_pred = cb.predict(X_test)
y_prob = cb.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
f1 = f1_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_prob)
r2 = r2_score(y_test, y_prob)

print("="*45)
print(f"Accuracy      : {acc*100:.2f}%")
print(f"AUC Score     : {auc:.4f}")
print(f"F1 Score      : {f1:.4f}")
print(f"MSE           : {mse:.4f}")
print(f"R Square      : {r2:.4f}")
print("="*45)

In [ ]:
import torch
torch.save(model.state_dict(), 'final_drug_discovery_deep_learning_model.pt')